In [0]:
# Uploading data into databricks
csv_data = """order_id,supplier_id,item_name,stock_level,delivery_date,status
101,5,Microchips Type-A,120,2026-06-15,Delivered
102,1,Aluminium Casings,5,null,Pending
103,2,Lithium Battery Packs,14,2026-06-28,Shipped
104,3,Copper Wiring Reels,200,2026-06-10,Delivered
105,4,Fiber Optic Transceivers,9,2026-06-01,Delivered
106,5,Microchips Type-B,45,2026-06-20,Delivered
107,2,Solid State Drives,3,2026-05-25,Delivered
108,1,Steel Brackets,88,null,Processing
109,3,Rubber Seals,350,2026-06-12,Delivered
110,4,Sensor Modules,12,2026-07-05,Shipped
111,5,Power Adapters,110,2026-06-18,Delivered
112,1,Cooling Fans,8,2026-06-05,Delivered
113,2,LED Displays,0,null,Pending
114,3,Capacitors Pack,500,2026-06-22,Delivered
115,4,Circuit Boards,19,2026-06-14,Delivered
"""
dbutils.fs.put("dbfs:/FileStore/tables/orders.csv", csv_data, overwrite=True)

Wrote 744 bytes.


True

In [0]:
# Extract data from databricks
df = spark.read.csv("dbfs:/FileStore/tables/orders.csv", header=True, inferSchema=True, nullValue="null")
display(df)

order_id,supplier_id,item_name,stock_level,delivery_date,status
101,5,Microchips Type-A,120,2026-06-15,Delivered
102,1,Aluminium Casings,5,null,Pending
103,2,Lithium Battery Packs,14,2026-06-28,Shipped
104,3,Copper Wiring Reels,200,2026-06-10,Delivered
105,4,Fiber Optic Transceivers,9,2026-06-01,Delivered
106,5,Microchips Type-B,45,2026-06-20,Delivered
107,2,Solid State Drives,3,2026-05-25,Delivered
108,1,Steel Brackets,88,null,Processing
109,3,Rubber Seals,350,2026-06-12,Delivered
110,4,Sensor Modules,12,2026-07-05,Shipped


In [0]:
# Transform data
from pyspark.sql.functions import *
df = df.na.drop(subset=["delivery_date"])
print("Data after dropping null values:")
display(df)

df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd"))
current_timeline = lit("2026-06-20")
df = df.withColumn("delay_days", datediff(current_timeline, col("delivery_date")))
df = df.withColumn("is_delayed", when(col("delay_days") > 0, "Yes").otherwise("No"))
print("Processed data:")
display(df)

Data after dropping null values:


order_id,supplier_id,item_name,stock_level,delivery_date,status
101,5,Microchips Type-A,120,2026-06-15,Delivered
103,2,Lithium Battery Packs,14,2026-06-28,Shipped
104,3,Copper Wiring Reels,200,2026-06-10,Delivered
105,4,Fiber Optic Transceivers,9,2026-06-01,Delivered
106,5,Microchips Type-B,45,2026-06-20,Delivered
107,2,Solid State Drives,3,2026-05-25,Delivered
109,3,Rubber Seals,350,2026-06-12,Delivered
110,4,Sensor Modules,12,2026-07-05,Shipped
111,5,Power Adapters,110,2026-06-18,Delivered
112,1,Cooling Fans,8,2026-06-05,Delivered


Processed data:


order_id,supplier_id,item_name,stock_level,delivery_date,status,delay_days,is_delayed
101,5,Microchips Type-A,120,2026-06-15,Delivered,5,Yes
103,2,Lithium Battery Packs,14,2026-06-28,Shipped,-8,No
104,3,Copper Wiring Reels,200,2026-06-10,Delivered,10,Yes
105,4,Fiber Optic Transceivers,9,2026-06-01,Delivered,19,Yes
106,5,Microchips Type-B,45,2026-06-20,Delivered,0,No
107,2,Solid State Drives,3,2026-05-25,Delivered,26,Yes
109,3,Rubber Seals,350,2026-06-12,Delivered,8,Yes
110,4,Sensor Modules,12,2026-07-05,Shipped,-15,No
111,5,Power Adapters,110,2026-06-18,Delivered,2,Yes
112,1,Cooling Fans,8,2026-06-05,Delivered,15,Yes


In [0]:
# Load data into Delta Table
df.write.format("delta").mode("overwrite").saveAsTable("orders_table")
print("Data saved into Delta Table.")

Data saved into Delta Table.


In [0]:

# Basic Analysis Queries
df.createOrReplaceTempView("orders")

print("Total delayed orders and Average delay days by Suppliers:")
display(spark.sql("""SELECT supplier_id, COUNT(order_id) as total_orders, COUNT(order_id) FILTER(WHERE is_delayed = 'Yes') as total_delayed_orders, 
    AVG(delay_days) as avg_delay_days FROM orders GROUP BY supplier_id ORDER BY total_delayed_orders DESC"""))

print("Low Stock Items:")
display(spark.sql("""SELECT item_name, stock_level FROM orders WHERE stock_level < 15 ORDER BY stock_level"""))

Total delayed orders and Average delay days by Suppliers:


supplier_id,total_orders,total_delayed_orders,avg_delay_days
5,3,2,2.3333333333333335
3,3,2,5.333333333333333
4,3,2,3.3333333333333335
1,1,1,15.0
2,2,1,9.0


Low Stock Items:


item_name,stock_level
Solid State Drives,3
Cooling Fans,8
Fiber Optic Transceivers,9
Sensor Modules,12
Lithium Battery Packs,14


In [0]:
print("OPTIMIZE orders")
display(spark.sql("OPTIMIZE orders_table"))

print("Z-ORDER BY supplier_id")
display(spark.sql("OPTIMIZE orders_table ZORDER BY (supplier_id)"))

print("VACUUM orders")
display(spark.sql("VACUUM orders_table"))

OPTIMIZE orders


path,metrics
abfss://unity-catalog-storage@dbstorage5fztkxjdbztzk.dfs.core.windows.net/7405613668275341/__unitystorage/catalogs/ade5da72-c7b5-4cb2-94f9-795b78c30995/tables/353e1771-2277-4ee3-a253-4fc7c24ff59c,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1782297410718, 1782297411258, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"


Z-ORDER BY supplier_id


path,metrics
abfss://unity-catalog-storage@dbstorage5fztkxjdbztzk.dfs.core.windows.net/7405613668275341/__unitystorage/catalogs/ade5da72-c7b5-4cb2-94f9-795b78c30995/tables/353e1771-2277-4ee3-a253-4fc7c24ff59c,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2861), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1782297412063, 1782297412474, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"


VACUUM orders


path
abfss://unity-catalog-storage@dbstorage5fztkxjdbztzk.dfs.core.windows.net/7405613668275341/__unitystorage/catalogs/ade5da72-c7b5-4cb2-94f9-795b78c30995/tables/353e1771-2277-4ee3-a253-4fc7c24ff59c
